In [5]:
import uproot
import pandas as pd
import numpy as np
import awkward as ak
import os
import json
import ROOT

In [6]:
os.environ['ttH_yy_DIR'] = '/eos/user/e/elmazzeo/ttH@FCC-hh/results/2025-03-21/'

In [26]:
basedir = os.path.join(os.environ.get('ttH_yy_DIR'), 'final')

process = {
    'ttHyy' : { 
        'sample_list' : ['mgp8_pp_tth01j_5f_84TeV_haaexcl'], 
        #'sample_list' : ['mgp8_pp_tth01j_5f_haa'], 
               'label' : r'$ttH$ $\rightarrow$ $\gamma\gamma$', 'kfactor' : 1.},
    'ttyy' : { 
        'sample_list' : ['mgp8_pp_ttaa01j_5f_84TeV'], 
        #'sample_list' : ['mgp8_pp_ttaa_semilep_5f_100TeV'], 
               'label' : '$tt\gamma\gamma$', 'kfactor' : 3.78},
    'Vyy_jets' : { 'sample_list' : ['mgp8_pp_Vaajj_HF_5f_84TeV'], 
               'label' : 'V$\gamma\gamma$+jets', 'kfactor' : 1.}
}

selection = {
#            "nocuts" : "All events", # all events
#            "photons": "$\geq$ 2 photons",
#            "photons_rel_pt": "Rel. $p_{T}$ cuts",
#            "photons_myy_window": "110 < $m_{\gamma\gamma}$ < 140 GeV",
#            "preselection": "$\geq$ 2 b-jets",
#            "lep_channel" : "$\geq$ 1 lepton",
            "pT_yy_bin1" : "0$\leq p_{T}(\gamma\gamma)$< 60 GeV",
            "pT_yy_bin2" : "60$\leq p_{T}(\gamma\gamma)$< 120 GeV",
            "pT_yy_bin3" : "120$\leq p_{T}(\gamma\gamma)$< 200 GeV",
            "pT_yy_bin4" : "200$\leq p_{T}(\gamma\gamma)$< 300 GeV",
            "pT_yy_bin5" : "300$\leq p_{T}(\gamma\gamma)$< 450 GeV",
            "pT_yy_bin6" : "$p_{T}(\gamma\gamma)\geq$ 450 GeV",

}

variables = ['weight']

process_infos = ["/eos/experiment/fcc/hh/utils/FCCDicts/FCChh_procDict_fcc_v07_II.json",
                 "/eos/experiment/fcc/hh/utils/FCCDicts/FCChh_procDict_fcc_v06_II.json"]

lumi = 3e7 # 3 * 10^7 pb-1 = 3 ab-1

In [27]:
process_dicts = []
for inputname in process_infos :
    with open(inputname, 'r') as f :
        process_dicts.append(json.load(f))

In [28]:
df = {}
sow = {}

In [29]:
for p in process.keys() :
    df[p] = {}
    print(p)
    for s in selection.keys() :
        df1 = []
        sow[p] = []
        print("\t"+s)
        for sample in process[p]['sample_list'] :
            print("\t\t"+sample)
            inputfile = os.path.join(basedir, sample+"_"+s+".root")
            # get sum of weights
            f = ROOT.TFile.Open(inputfile)
            sow[p].append(f.Get("SumOfWeights").GetVal())
            f.Close()
            if sample == 'mgp8_pp_tth01j_5f_haa' :
                sow[p][-1] = 1690468.0 - 117.0
            # get sample dict
            for d in process_dicts :
                if sample in list(d.keys()) :
                    process_dict = d.copy()
                    break
            # get sample
            with uproot.open(inputfile) as f :
                df1.append(ak.to_dataframe(f['events'].arrays(expressions=variables, library='ak')))
                df1[-1]["weight"] = df1[-1]["weight"]/sow[p][-1]*process_dict[sample]['crossSection']*process_dict[sample]['kfactor']*process_dict[sample]['matchingEfficiency']
        df[p][s] = pd.concat(df1, copy=True, ignore_index=True)
        df[p][s]['weight'] = df[p][s]['weight']*process[p]['kfactor']

ttHyy
	pT_yy_bin1
		mgp8_pp_tth01j_5f_84TeV_haaexcl
	pT_yy_bin2
		mgp8_pp_tth01j_5f_84TeV_haaexcl
	pT_yy_bin3
		mgp8_pp_tth01j_5f_84TeV_haaexcl
	pT_yy_bin4
		mgp8_pp_tth01j_5f_84TeV_haaexcl
	pT_yy_bin5
		mgp8_pp_tth01j_5f_84TeV_haaexcl
	pT_yy_bin6
		mgp8_pp_tth01j_5f_84TeV_haaexcl
ttyy
	pT_yy_bin1
		mgp8_pp_ttaa01j_5f_84TeV
	pT_yy_bin2
		mgp8_pp_ttaa01j_5f_84TeV
	pT_yy_bin3
		mgp8_pp_ttaa01j_5f_84TeV
	pT_yy_bin4
		mgp8_pp_ttaa01j_5f_84TeV
	pT_yy_bin5
		mgp8_pp_ttaa01j_5f_84TeV
	pT_yy_bin6
		mgp8_pp_ttaa01j_5f_84TeV
Vyy_jets
	pT_yy_bin1
		mgp8_pp_Vaajj_HF_5f_84TeV
	pT_yy_bin2
		mgp8_pp_Vaajj_HF_5f_84TeV
	pT_yy_bin3
		mgp8_pp_Vaajj_HF_5f_84TeV
	pT_yy_bin4
		mgp8_pp_Vaajj_HF_5f_84TeV
	pT_yy_bin5
		mgp8_pp_Vaajj_HF_5f_84TeV
	pT_yy_bin6
		mgp8_pp_Vaajj_HF_5f_84TeV


In [30]:
my_entries = {
    "Selection" : []
}

In [31]:
for p in process.keys() :
    my_entries[process[p]['label']] = []

In [32]:
for s in selection :
    my_entries["Selection"].append(selection[s])
    for p in process.keys() :
        my_entries[process[p]['label']].append(len(df[p][s]))

In [33]:
my_entries = pd.DataFrame(my_entries)
my_entries = my_entries.set_index('Selection')

In [34]:
my_entries

,$ttH$ $\rightarrow$ $\gamma\gamma$,$tt\gamma\gamma$,V$\gamma\gamma$+jets
Selection,,,
0$\leq p_{T}(\gamma\gamma)$< 60 GeV,100560,12936,2954
60$\leq p_{T}(\gamma\gamma)$< 120 GeV,171022,18467,2889
120$\leq p_{T}(\gamma\gamma)$< 200 GeV,170794,11914,1145
200$\leq p_{T}(\gamma\gamma)$< 300 GeV,114825,5040,312
300$\leq p_{T}(\gamma\gamma)$< 450 GeV,71279,1896,96
$p_{T}(\gamma\gamma)\geq$ 450 GeV,43295,683,20


In [35]:
my_yields = {
    "Selection" : []
}

In [36]:
for p in process.keys() :
    my_yields[process[p]['label']] = []

In [37]:
for s in selection :
    my_yields["Selection"].append(selection[s])
    for p in process.keys() :
        my_yields[process[p]['label']].append(df[p][s]["weight"].sum()*lumi)

In [38]:
my_yields = pd.DataFrame(my_yields)
my_yields = my_yields.set_index('Selection')

In [39]:
my_yields

,$ttH$ $\rightarrow$ $\gamma\gamma$,$tt\gamma\gamma$,V$\gamma\gamma$+jets
Selection,,,
0$\leq p_{T}(\gamma\gamma)$< 60 GeV,18216.399698,44044.727453,872.854379
60$\leq p_{T}(\gamma\gamma)$< 120 GeV,30980.176255,62876.495678,853.641817
120$\leq p_{T}(\gamma\gamma)$< 200 GeV,30938.585380,40568.973562,338.255462
200$\leq p_{T}(\gamma\gamma)$< 300 GeV,20798.701776,17160.266755,92.175438
300$\leq p_{T}(\gamma\gamma)$< 450 GeV,12911.664392,6457.246348,28.360564
$p_{T}(\gamma\gamma)\geq$ 450 GeV,7841.875690,2325.970170,5.915264


In [21]:
my_eff = {
    "Selection" : []
}

In [22]:
for p in process.keys() :
    my_eff[process[p]['label']] = []

In [23]:
for s in selection :
    my_eff["Selection"].append(selection[s])
    for p in process.keys() :
        my_eff[process[p]['label']].append(df[p][s]["weight"].sum()/df[p]["nocuts"]["weight"].sum())

In [24]:
my_eff = pd.DataFrame(my_eff)
my_eff = my_eff.set_index('Selection')

In [25]:
my_eff

,$ttH$ $\rightarrow$ $\gamma\gamma$,$tt\gamma\gamma$,V$\gamma\gamma$+jets
Selection,,,
All events,1.000000,1.000000,1.000000
$\geq$ 2 photons,0.481783,0.202914,0.169577
Rel. $p_{T}$ cuts,0.430959,0.153124,0.115458
110 < $m_{\gamma\gamma}$ < 140 GeV,0.421720,0.106801,0.076335
$\geq$ 2 b-jets,0.250833,0.063970,0.004793
$\geq$ 1 lepton,0.072244,0.018333,0.002853
